In [ ]:
!wget https://raw.githubusercontent.com/mainlp/xsid/refs/heads/main/data/xSID-0.7/en.train.conll

In [ ]:
from collections import defaultdict

In [ ]:
def parse_examples(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    return content.strip().split('\n\n')

In [ ]:
def extract_key_info(example):
    text = None
    intent = None
    for line in example.split('\n'):
        if line.startswith('# text = '):
            text = line.replace('# text = ', '').strip()
        elif line.startswith('# intent ='):
            intent = line.replace('# intent =', '').strip()
    return text, intent

In [ ]:
def get_unique_examples(examples):
    seen_keys = set()
    unique_examples = []

    for example in examples:
        text, intent = extract_key_info(example)
        key = (text, intent)

        if key not in seen_keys:
            seen_keys.add(key)
            unique_examples.append(example)

    return unique_examples

In [ ]:
def process_and_save(examples, output_file):
    processed_examples = []

    for idx, example in enumerate(examples, 1):
        lines = example.split('\n')
        new_lines = []

        # Add ID at the beginning
        new_lines.append(f'# id: train_{idx}')

        for line in lines:
            # Skip the slots line
            if line.startswith('# slots:'):
                continue

            # Keep text and intent headers unchanged
            if line.startswith('#'):
                new_lines.append(line)
                continue

            # Process token lines (words)
            # Format: "1 word intent slot" -> convert to "1 word slot"
            parts = line.strip().split()
            if len(parts) >= 4: # If there is an index, word, intent, slot
                # parts[0] = id (1)
                # parts[1] = word (show)
                # parts[2] = intent (reminder/...) -> REMOVE THIS
                # parts[-1] = slot (O or B-reference) -> KEEP THIS

                # Reassemble: index, word, slot
                clean_line = f"{parts[0]} {parts[1]} {parts[-1]}"
                new_lines.append(clean_line)
            else:
                # If the line is unusual, keep it as is (just in case)
                if line.strip():
                    new_lines.append(line)

        processed_examples.append('\n'.join(new_lines))

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(processed_examples))
        f.write('\n\n')

In [ ]:
def save_full_reference(examples, output_file):
    """
    Save FULL original examples with IDs.
    Nothing is removed (slots and intent lines are preserved).
    Needed for restoring data after translation.
    """
    processed = []
    for idx, example in enumerate(examples, 1):
        # Simply add ID to the original example
        processed.append(f"# id: train_{idx}\n{example}")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(processed))
        f.write('\n\n')

In [ ]:
raw_examples = parse_examples('en.train.conll')
unique_examples = get_unique_examples(raw_examples)

# 1. File for the model (cleaned)
process_and_save(unique_examples, 'en.train.unique_ids.conll')

# 2. Reference file (original with IDs for restoration)
save_full_reference(unique_examples, 'en.train.reference.conll')